# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [18]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [19]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [20]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table':["Employees","Department","Salary"], # ENTER A TABLE COLUMNS HERE,
        'definition': ["Employee information","Employee Department information","Employee Salary information"] # ENTER A TABLE DEFINITATIONS HERE
        }
df = pd.DataFrame(data)
print(df)

        table                       definition
0   Employees             Employee information
1  Department  Employee Department information
2      Salary      Employee Salary information


In [21]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [22]:
print(text_tables)

Employees: Employee information
Department: Employee Department information
Salary: Employee Salary information


In [23]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [24]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Write a query to give maximum salary of employee group by department")

In [25]:
print(return_OAI(pqt1))

```json
{
    "tables": {
        "Employees": "Employee information",
        "Department": "Employee Department information",
        "Salary": "Employee Salary information"
    }
}
```


In [26]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="how many emplyees in each department")

In [27]:
print(return_OAI(pqt3))

```json
{
    "tables": ["Employees", "Department"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [28]:
pqt4 = prompt_question_tables.format(tables=text_tables,
                                     question="count the number of employees in each department")

In [29]:
print(return_OAI(pqt3))

```json
{
    "tables": ["Employees", "Department"]
}
```


Learning:
Use the OpenAI GPT-3.5-turbo model to identify relevant SQL tables for a given user query.
Define a set of tables with their descriptions in a pandas DataFrame.
Format a prompt for the language model, including table definitions and a user question.
Parse the model's response, which returns the selected table names in a JSON format.